# Notebook 03 — Ablation Study

Systematic ablation comparing Models A / B / C with and without each component,
following the evaluation rubric requirements.

## Ablation Table (required deliverable)

| | Model A — ML Only | Model B — DL Only | Model C — Hybrid |
|---|---|---|---|
| TF-IDF + Jaccard | ✓ | ✗ | ✓ (0.20 weight) |
| SBERT bi-encoder | ✗ | ✓ (0.30) | ✗ |
| Cross-encoder | ✗ | ✓ (0.70) | ✓ (0.50) |
| BiLSTM quality | ✗ | ✗ | ✓ (0.30) |

## Metrics used
- **Pearson r**: correlation with human-assigned grades (simulated)
- **MAE**: mean absolute error vs human grades (0-10 scale)
- **Rank accuracy**: proportion of pairs ranked correctly (high > medium > low)

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'engine'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import pearsonr

from answer_scorer import (
    ml_similarity, dl_similarity, hybrid_similarity,
    similarity_to_marks, GradingMode, run_ablation
)

## 1. Simulated Ground-Truth Dataset

In [ ]:
# Each row: (teacher_answer, student_answer, human_grade_out_of_10)
EVAL_SET = [
    ('Photosynthesis uses sunlight, water, and CO2 to produce glucose and oxygen.',
     'Plants use sunlight and CO2 to make sugar and release O2 through photosynthesis.',
     9.0),
    ('Photosynthesis uses sunlight, water, and CO2 to produce glucose and oxygen.',
     'Photosynthesis is a process done by plants.',
     4.0),
    ('Photosynthesis uses sunlight, water, and CO2 to produce glucose and oxygen.',
     'The cell cycle involves mitosis and meiosis for reproduction.',
     0.5),
    ("Newton's second law: F = ma, where F is force, m is mass, a is acceleration.",
     'Force equals mass multiplied by acceleration according to Newton.',
     8.5),
    ("Newton's second law: F = ma, where F is force, m is mass, a is acceleration.",
     'Newton said that moving things need force to stop.',
     3.5),
    ('Osmosis is the movement of water across a semi-permeable membrane from low to high solute concentration.',
     'Water moves through a membrane from dilute to concentrated solution — this is osmosis.',
     9.0),
    ('Osmosis is the movement of water across a semi-permeable membrane from low to high solute concentration.',
     'Osmosis is when water moves.',
     2.0),
    ('The French Revolution began in 1789, overthrowing the monarchy and establishing republic ideals.',
     'The French revolution happened in 1789 and removed the king, bringing new ideas about freedom and equality.',
     8.0),
    ('The French Revolution began in 1789, overthrowing the monarchy and establishing republic ideals.',
     'France had a war a long time ago.',
     1.0),
    ('DNA replication is semi-conservative: each new double helix contains one original and one new strand.',
     'In DNA replication, the two strands unzip and each serves as a template, producing two copies each with one old and one new strand.',
     9.5),
]

teachers    = [r[0] for r in EVAL_SET]
students    = [r[1] for r in EVAL_SET]
human_marks = [r[2] for r in EVAL_SET]

print(f'Evaluation set: {len(EVAL_SET)} answer pairs')
print(f'Human grade range: {min(human_marks)} – {max(human_marks)}')

## 2. Compute All Model Predictions

In [ ]:
ml_preds     = [similarity_to_marks(ml_similarity(s, t))     for s, t in zip(students, teachers)]
dl_preds     = [similarity_to_marks(dl_similarity(s, t))     for s, t in zip(students, teachers)]
hybrid_preds = [similarity_to_marks(hybrid_similarity(s, t)) for s, t in zip(students, teachers)]

results_df = pd.DataFrame({
    'Teacher': [t[:40]+'...' for t in teachers],
    'Student' : [s[:40]+'...' for s in students],
    'Human'   : human_marks,
    'Model A (ML)'    : ml_preds,
    'Model B (DL)'    : dl_preds,
    'Model C (Hybrid)': hybrid_preds,
})
print(results_df[['Human','Model A (ML)','Model B (DL)','Model C (Hybrid)']].to_string())

## 3. Ablation Metrics

In [ ]:
def compute_metrics(preds, human):
    mae = np.mean(np.abs(np.array(preds) - np.array(human)))
    r, p = pearsonr(preds, human)
    return {'MAE': round(mae, 3), 'Pearson_r': round(r, 3), 'p_value': round(p, 4)}

metrics = {
    'Model A — ML Only'  : compute_metrics(ml_preds,     human_marks),
    'Model B — DL Only'  : compute_metrics(dl_preds,     human_marks),
    'Model C — Hybrid'   : compute_metrics(hybrid_preds, human_marks),
}

print('\n' + '='*55)
print(f'{"Model":<22} {"MAE":>6}  {"Pearson r":>10}  {"p-value":>9}')
print('-'*55)
for name, m in metrics.items():
    print(f'{name:<22} {m["MAE"]:>6.3f}  {m["Pearson_r"]:>10.3f}  {m["p_value"]:>9.4f}')
print('='*55)
print('(Lower MAE = better; Higher Pearson r = better)')

## 4. Ablation Table Plot

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
x = np.arange(len(EVAL_SET))

for ax, (preds, label, color) in zip(axes, [
    (ml_preds,     'Model A — ML',     '#f59e0b'),
    (dl_preds,     'Model B — DL',     '#3b82f6'),
    (hybrid_preds, 'Model C — Hybrid', '#10b981'),
]):
    ax.scatter(human_marks, preds, color=color, s=80, zorder=5, label=label)
    ax.plot([0, 10], [0, 10], 'k--', alpha=.3, linewidth=1)
    m = compute_metrics(preds, human_marks)
    ax.set_title(f'{label}\nMAE={m["MAE"]}  r={m["Pearson_r"]}', fontsize=11)
    ax.set_xlabel('Human Grade')
    ax.set_ylabel('Predicted Grade')
    ax.set_xlim(0, 10.5)
    ax.set_ylim(0, 10.5)
    ax.grid(alpha=.2)

plt.suptitle('Ablation Study: Human vs Predicted Grades', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Component Contribution Analysis

In [ ]:
from answer_scorer import _tfidf_cosine, _jaccard_keyword, _cross_encoder_score, _sbert_cosine
from lstm_quality  import NeuralQualityScorer

qs = NeuralQualityScorer()

component_scores = []
for s, t in zip(students[:5], teachers[:5]):  # first 5 pairs for speed
    component_scores.append({
        'TF-IDF'         : round(_tfidf_cosine(s, t), 3),
        'Jaccard'        : round(_jaccard_keyword(s, t), 3),
        'Cross-encoder'  : round(_cross_encoder_score(s, t), 3),
        'SBERT bi-enc'   : round(_sbert_cosine(s, t), 3),
        'BiLSTM quality' : round(qs.score(s, t), 3),
    })

comp_df = pd.DataFrame(component_scores, index=[f'Pair {i+1}' for i in range(5)])
print(comp_df.to_string())

# Correlation of each component with human grades (first 5)
print('\nCorrelation with human grades (first 5 pairs):')
for col in comp_df.columns:
    r, _ = pearsonr(comp_df[col], human_marks[:5])
    print(f'  {col:20s}: r={r:.3f}')

## 6. Learning Curve Simulation

Shows how adding more answer pairs affects the stability of Pearson r.

In [ ]:
# Simulate by sub-sampling EVAL_SET
np.random.seed(42)
sizes  = list(range(3, len(EVAL_SET) + 1))
r_ml, r_dl, r_hy = [], [], []

for n in sizes:
    idx = np.random.choice(len(EVAL_SET), n, replace=False)
    sub_ml  = [ml_preds[i]     for i in idx]
    sub_dl  = [dl_preds[i]     for i in idx]
    sub_hy  = [hybrid_preds[i] for i in idx]
    sub_hum = [human_marks[i]  for i in idx]
    r_ml.append(pearsonr(sub_ml,  sub_hum)[0])
    r_dl.append(pearsonr(sub_dl,  sub_hum)[0])
    r_hy.append(pearsonr(sub_hy,  sub_hum)[0])

plt.figure(figsize=(9, 4))
plt.plot(sizes, r_ml, 'o-', color='#f59e0b', label='Model A — ML')
plt.plot(sizes, r_dl, 's-', color='#3b82f6', label='Model B — DL')
plt.plot(sizes, r_hy, '^-', color='#10b981', label='Model C — Hybrid')
plt.xlabel('Number of evaluation pairs')
plt.ylabel('Pearson r with human grades')
plt.title('Stability of Model Agreement with Human Graders')
plt.legend()
plt.grid(alpha=.2)
plt.tight_layout()
plt.show()

## 7. Final Ablation Summary Table

In [ ]:
summary = pd.DataFrame([
    {'Model': 'A — ML Only',   'TF-IDF': '✓', 'Jaccard': '✓', 'SBERT': '✗', 'Cross-enc': '✗', 'BiLSTM': '✗',
     'MAE': metrics['Model A — ML Only']['MAE'],
     'Pearson r': metrics['Model A — ML Only']['Pearson_r']},
    {'Model': 'B — DL Only',   'TF-IDF': '✗', 'Jaccard': '✗', 'SBERT': '✓', 'Cross-enc': '✓', 'BiLSTM': '✗',
     'MAE': metrics['Model B — DL Only']['MAE'],
     'Pearson r': metrics['Model B — DL Only']['Pearson_r']},
    {'Model': 'C — Hybrid ★',  'TF-IDF': '✓', 'Jaccard': '✗', 'SBERT': '✗', 'Cross-enc': '✓', 'BiLSTM': '✓',
     'MAE': metrics['Model C — Hybrid']['MAE'],
     'Pearson r': metrics['Model C — Hybrid']['Pearson_r']},
])
summary = summary.set_index('Model')
print(summary.to_string())
print('\n★ = Proposed System (Hybrid)')
print('Lower MAE is better; Higher Pearson r is better.')